In [ ]:
import pandas as pd
import numpy as np
import math as mt
import os
import pandas as pd
import matplotlib.pyplot as plt
import sys
import csv
import traceback

sys.path.append(os.path.abspath('..'))  # REDLAB_UQ.py is in the project root


def mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean((y_true - y_pred) ** 2)


def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot == 0:
        return 1.0 if ss_res == 0 else 0.0
    return 1.0 - (ss_res / ss_tot)

# Load (and reload) the REDLAB_UQ module
try:
    from REDLAB_UQ import SurrogateModel
    # Reload the module if it is already loaded
    import REDLAB_UQ
    import importlib
    importlib.reload(REDLAB_UQ)
    from REDLAB_UQ import SurrogateModel
    print("REDLAB_UQ module loaded.")
except ImportError as e:
    print(f"Failed to load the REDLAB_UQ module: {e}")
    print("Check that REDLAB_UQ.py is in the project root folder.")
except Exception as e:
    print(f"Unexpected error: {e}")

## Import the UQ class
model = SurrogateModel()

In [ ]:


## Mean value
#base_values = [15] * 10

#base_values = [26.894765993870866, 26.78963336562726, 60.5466610514048, 7.870646571918311, 70.00000019778525, 304.9999999981768] # optimal, sample 13
#base_values = [28.108544215665155, 26.460874012505595, 61.25694096258407, 7.344594158921096, 69.99999990493262, 305.00000005519576] # sample 4
base_values = [36, 36, 70, 5, 70, 305] # initial, sample 1

#base_values = [27.018145625189476, 26.57140940179377, 60.450871383019354, 7.716333102378568, 70.00000014854835, 305.0000000392265] # restart sample 2
#base_values = [27.006769463818138, 26.580683617926113, 60.44175843235475, 7.721046448633007, 70.00000014724861, 305.0000000389639] # restart sample 2
#base_values = [27.016485865613774, 26.572762491692053, 60.44954182143573, 7.7170207703294755, 70.00000014835872, 305.0000000391882] # restart sample 2

# Skipped sample indices
#skipped_sample_indices = [] # optimal
skipped_sample_indices = [73] # initial


# Load the CSV files
df_inputs_z = pd.read_csv("../initial/input_samples_zz.csv")    # normalized input samples (generated by sample_generator_project2.ipynb)
scaled_inputs = df_inputs_z.to_numpy(dtype=float) * np.array(base_values, dtype=float)
df_inputs = pd.DataFrame(scaled_inputs, columns=df_inputs_z.columns)

# Prepare the input and output data
input_samples = df_inputs_z.to_numpy()

# # Save input_samples to a separate CSV file
# df_inputs.to_csv(input_samples_csv_path, index=False)
# print(f"✅ input_samples saved to: {input_samples_csv_path}")

# Load the output data from CSV
df_values = pd.read_csv("../initial/sample_1.csv") # initial design

#df_values = pd.read_csv("../samples/260821_225714/sample_10.csv")
#df_values = pd.read_csv("../samples/260821_225714/sample_17.csv")


output_samples = df_values.to_numpy()


In [ ]:
N = 6  # number of variables
p = 2  # maximum polynomial order (m)
S = 1

option = 'SLS'
#option = 'Lasso'

basis_num = model.calculate_basis_num(N,p,S)
print("Basis num: ", basis_num)
print("Required samples num, 2 product:", int(basis_num * 2))

if option == 'SLS':
    min_samples = min(int(basis_num * 2), output_samples.shape[0], input_samples.shape[0])
elif option == 'Lasso':
    min_samples = min(int(basis_num * 1), output_samples.shape[0], input_samples.shape[0])
else:
    pass # error

# Filter the data according to skipped_sample_indices
# Convert the 1-based indices to 0-based and store them in a set
skipped_indices_0based = set(idx - 1 for idx in skipped_sample_indices)

# Collect the valid indices sequentially
valid_indices = []
idx = 0
while len(valid_indices) < min_samples:
    if idx not in skipped_indices_0based:
        valid_indices.append(idx)
    idx += 1

print(f"Number of rows in the data: {input_samples.shape[0]}")
print(f"Skipped indices (1-based): {skipped_sample_indices}")
print(f"Skipped indices (0-based): {list(skipped_indices_0based)}")
print(f"Required number of valid samples: {min_samples}")
print(f"Number of valid indices: {len(valid_indices)}")
print(f"End of the required range: {idx}")

# Select the samples with the valid indices
valid_indices = np.array(valid_indices)
input_samples_coeff = input_samples[valid_indices, :]
output_samples = output_samples[valid_indices, :]

print("Samples num: ", min_samples)
print("inputs shape:", input_samples_coeff.shape)  # numbers of rows and columns

# Filter the outputs of a given column by the condition
target_values = [0,1,2,3]
# target_values = [20, 21, 22, 24, 25, 26, 28, 29, 30, 52, 53, 54, 56, 57, 58, 60, 61, 62]

output_filtered = np.zeros((output_samples.shape[0], len(target_values)))

index = 0
for i in target_values:
    output_filtered[:,index] = output_samples[:,i]
    index = index+1

output_samples = output_filtered

# Show part of the outputs for checking
print("✅ output_samples shape:", output_samples.shape)
print("✅ First 5 rows of output_samples:")
print(output_samples[:5])  # first 5 rows

In [ ]:
## Normalized input samples

input_means = np.array(base_values)
input_normalize = input_samples
input_normalize_coeff = input_samples_coeff

## Multiply input_samples and input_samples_coeff by the mean to obtain physical values
input_samples = input_normalize * input_means
input_samples_coeff = input_normalize_coeff * input_means

print("Normalized inputs (first 5 rows):")
print(input_normalize_coeff[:5, :])

## GPCE Algorithm

In [ ]:
# Samples with L rows (samples) and N columns (basis functions)
L = input_samples.shape[0]
print(L)

# Build the monomial basis
train_basis = model.P_grlex_n_DD_fast(input_normalize, p, S)
train_basis_coeff = train_basis[:, :input_samples_coeff.shape[0]]

W3_DD = model.whitening_transform(train_basis,L)

# # Test code
# PT = np.transpose(train_basis)
# G = (train_basis @ PT) / L

# # Eigenvalues of G
# eigvals = np.linalg.eigvals(G)

# # Filter negative or near-zero eigenvalues (numerical error)
# negative_eigvals = eigvals[eigvals < -1e-10]  # -1e-10: threshold for numerical stability

# # Output
# print("Negative eigenvalues:")
# print(negative_eigvals)
# print(f"{len(negative_eigvals)} negative or non-definite eigenvalues.")




## DD-GPCE coefficient estimation
A3T_DD=W3_DD@train_basis_coeff
A3_DD=np.transpose(A3T_DD)

B3_DD=output_samples
C3_DD=np.linalg.inv(A3T_DD@A3_DD)@A3T_DD@B3_DD

print(C3_DD.shape[0], C3_DD.shape[1])

## Lasso regression
if option == "Lasso":
    C3_DD_lasso = model.estimate_lasso_coefficients(A3_DD, B3_DD, alpha=1e-3)

    print(np.mean(abs(C3_DD_lasso-C3_DD)))
    print(abs(C3_DD_lasso-C3_DD))

### Test Code ###
if option == "Lasso":
    C3_DD = C3_DD_lasso
    print("Lasso regression coefficients used.")


# ## D-MORPH
# L_DMORPH = 80
# sampler2 = qmc.Sobol(d=N, scramble=True)
# X_uniform_DMORPH = sampler2.random(L_DMORPH + 2)[2:] # discard the first two points
# train3_samples_DMORPH = qmc.scale(X_uniform_DMORPH, l_bounds=[-np.pi] * N, u_bounds=[np.pi] * N)


In [ ]:
# DD-GPCE predictions for all input samples
print("DD-GPCE predictions for all input samples (df_inputs_z)")
print("="*70)

# Normalized input data
all_inputs_normalize = df_inputs_z.to_numpy()

# Build the basis
print(f"Building the basis ({all_inputs_normalize.shape[0]} samples)...")
all_basis = model.P_grlex_n_DD_fast(all_inputs_normalize, p, S)
print(f"Basis built (shape: {all_basis.shape})")

# Apply the whitening transformation and predict
print(f"Predicting the outputs...")
A_all_test = np.transpose(W3_DD @ all_basis)
Y_all_predicted = model.PCE(C3_DD, A_all_test)
print(f"Prediction completed (shape: {Y_all_predicted.shape})")

# Print the statistics
print(f"\nStatistics of the predicted outputs for all inputs:")
for i in range(len(target_values)):
    mean_val = np.mean(Y_all_predicted[:, i])
    std_val = np.std(Y_all_predicted[:, i])
    min_val = np.min(Y_all_predicted[:, i])
    max_val = np.max(Y_all_predicted[:, i])
    print(f"Output {i+1}: Mean={mean_val:.6f}, Std={std_val:.6f}, Min={min_val:.6f}, Max={max_val:.6f}")

# Histograms
print(f"\nPlotting histograms...")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

n_bins = 200

for i in range(len(target_values)):
    ax = axes[i]
    
    data = Y_all_predicted[:, i]
    
    # Histograms
    ax.hist(data, bins=n_bins, alpha=0.7, color='#1f77b4', 
            density=True, edgecolor='black', linewidth=0.5)
    
    # Statistics
    mean_val = np.mean(data)
    std_val = np.std(data)
    median_val = np.median(data)
    
    # Mean lines
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {mean_val:.2f}')
    ax.axvline(median_val, color='green', linestyle='--', linewidth=2, 
               label=f'Median: {median_val:.2f}')
    
    # Title and axis labels
    ax.set_title(f'Output {i+1} Distribution\n(n={data.shape[0]:,}, μ={mean_val:.2f}, σ={std_val:.2f})', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('Value', fontsize=12)
    ax.set_ylabel('Probability Density', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='both', labelsize=11)

plt.tight_layout()

# Save as an image file
plt.savefig('all_input_prediction_distribution.png', dpi=300, bbox_inches='tight')
print("\nHistograms saved to 'all_input_prediction_distribution.png'.")

plt.show()

print("="*70)
print("Prediction and visualization for all input samples completed.")

In [ ]:
# Failure probabilities of Outputs 2, 3, and 4 from the 1,000,000 samples
print("📊 Failure Probability Analysis (1,000,000 samples)")
print("="*70)

# Failure conditions
# Output 2 (index 1): failure if > 0.05
# Output 3 (index 2): failure if < 10
# Output 4 (index 3): failure if < 100

failure_conditions = {
    'Output 2': {'index': 1, 'condition': 'greater', 'threshold': 0.05},
    'Output 3': {'index': 2, 'condition': 'less', 'threshold': 10},
    'Output 4': {'index': 3, 'condition': 'less', 'threshold': 100}
}

# Failure probability
failure_results = []

print("\n📈 Failure Probability Results:")
print("-" * 70)

for output_name, criteria in failure_conditions.items():
    idx = criteria['index']
    threshold = criteria['threshold']
    condition = criteria['condition']
    
    # Number of failed samples
    if condition == 'greater':
        failure_count = np.sum(Y_all_predicted[:, idx] > threshold)
        condition_text = f"> {threshold}"
    else:  # less
        failure_count = np.sum(Y_all_predicted[:, idx] < threshold)
        condition_text = f"< {threshold}"
    
    n_large_samples = 1000_000
    # Failure probability
    failure_probability = failure_count / n_large_samples
    
    # Reliability (probability of safety)
    reliability = 1 - failure_probability
    
    print(f"\n{output_name}:")
    print(f"  Failure Condition: {condition_text}")
    print(f"  Failure Count: {failure_count:,} / {n_large_samples:,}")
    print(f"  Failure Probability: {failure_probability:.6f} ({failure_probability*100:.4f}%)")
    print(f"  Reliability: {reliability:.6f} ({reliability*100:.4f}%)")
    
    failure_results.append({
        'Output': output_name,
        'Condition': condition_text,
        'Failure_Count': failure_count,
        'Total_Samples': n_large_samples,
        'Failure_Probability': failure_probability,
        'Failure_Probability_%': failure_probability * 100,
        'Reliability': reliability,
        'Reliability_%': reliability * 100
    })

# Collect the results in a DataFrame
failure_df = pd.DataFrame(failure_results)
print("\n" + "="*70)
print("📋 Failure Probability Summary Table:")
print(failure_df.round(6))

# Visualization - failure probabilities
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Bar chart of the failure probabilities
outputs = failure_df['Output'].values
pof = failure_df['Failure_Probability'].values
reliability = failure_df['Reliability'].values

ax1.bar(outputs, pof * 100, color='#ff6b6b', alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_title('Failure Probability (%)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Probability (%)', fontsize=12)
ax1.set_ylim(0, max(pof * 100) * 1.2 if max(pof) > 0 else 1)
ax1.grid(True, alpha=0.3, axis='y')

# Show the value above each bar
for i, (output, prob) in enumerate(zip(outputs, pof)):
    ax1.text(i, prob * 100 + max(pof * 100) * 0.02, f'{prob*100:.4f}%', 
             ha='center', va='bottom', fontsize=11, fontweight='bold')

# Bar chart of the reliability
ax2.bar(outputs, reliability * 100, color='#51cf66', alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_title('Reliability (1 - Failure Probability) (%)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Reliability (%)', fontsize=12)
ax2.set_ylim(0, 105)
ax2.grid(True, alpha=0.3, axis='y')

# Show the value above each bar
for i, (output, rel) in enumerate(zip(outputs, reliability)):
    ax2.text(i, rel * 100 + 0.5, f'{rel*100:.4f}%', 
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("="*70)
print("Failure probability analysis completed.")

## Model Test

In [ ]:
from sklearn.metrics import r2_score

# Number of test samples
n_test_samples = 100  # number of test samples

# Load the CSV files

test_samples = df_inputs_z.to_numpy()
output_samples = df_values.to_numpy()

# Filter the test data according to skipped_sample_indices
# Convert the 1-based indices to 0-based and store them in a set
skipped_indices_0based = set(idx - 1 for idx in skipped_sample_indices)

# The test samples start after the training indices
test_start_idx = int(np.max(valid_indices)) + 1

# The number of test samples counts only the additional test samples
n_test_total = n_test_samples
n_total_rows = min(test_samples.shape[0], output_samples.shape[0])

valid_indices_test = []
idx = test_start_idx

while len(valid_indices_test) < n_test_total and idx < n_total_rows:
    if idx not in skipped_indices_0based:
        valid_indices_test.append(idx)
    idx += 1

print(f"Number of rows in the data: {n_total_rows}")
print(f"Skipped indices (1-based): {skipped_sample_indices}")
print(f"Requested number of samples: {n_test_total} (training {min_samples} + test {n_test_samples})")
print(f"Number of valid indices: {len(valid_indices_test)}")
if len(valid_indices_test) < n_test_total:
    print(f"Warning: not enough valid samples; only {len(valid_indices_test)} are used for testing.")

# Keep only the valid indices
valid_indices_test = np.array(valid_indices_test)
test_samples = test_samples[valid_indices_test, :]
output_samples = output_samples[valid_indices_test, :]

print(f"Number of test samples: {test_samples.shape[0]} (requested: {n_test_total})")
print(f"   Training range: [0:{int(valid_indices[-1]) + 1}], test range: [{valid_indices_test[0]}:{valid_indices_test[-1] + 1}]")

# Normalization
test_samples_normalize = test_samples
test_samples = test_samples_normalize * input_means # physical values
#test_samples_normalize = test_samples - input_means

# Use all input data
test_basis = model.P_grlex_n_DD_fast(test_samples_normalize,p, S)


output_filtered_test = np.zeros((output_samples.shape[0], len(target_values)))

index = 0
for i in target_values:
    output_filtered_test[:,index] = output_samples[:,i]
    index = index+1

output_samples = output_filtered_test

YY = output_samples
print(YY.shape[0], YY.shape[1])

A3T_test_DD=W3_DD@test_basis
A3_test_DD=np.transpose(A3T_test_DD)

print("C3: ", C3_DD.shape[0], C3_DD.shape[1])
print("A3_test_DD: ", A3_test_DD.shape[0], A3_test_DD.shape[1])
Yhat_DD_GPCE = model.PCE(C3_DD,A3_test_DD)


In [ ]:
## Plot
col_plot = int(np.sqrt(len(target_values)))
while len(target_values) % col_plot != 0:
    col_plot -= 1
row_plot = len(target_values) // col_plot

fig, axes = plt.subplots(row_plot, col_plot, figsize=(col_plot * 5, row_plot * 5))
axes = axes.flatten()

r2_data =[]
rmse_data =[]
adj_r2_data = []  # adjusted R²
for i in range(len(target_values)):
    ax = axes[i]
    ax.scatter(Yhat_DD_GPCE[:, i], YY[:, i], color='#FF7F00', label='DD-GPCE Model', alpha=0.6)
    
    # Compute the common range of the x and y axes
    x_min, x_max = Yhat_DD_GPCE[:, i].min(), Yhat_DD_GPCE[:, i].max()
    y_min, y_max = YY[:, i].min(), YY[:, i].max()
    
    # Minimum and maximum over the whole range
    overall_min = min(x_min, y_min)
    overall_max = max(x_max, y_max)
    
    # Draw the y=x line (same range)
    ax.plot([overall_min, overall_max], [overall_min, overall_max], 'k--', label='y = x')
    
    # Use the same range for the x and y axes
    ax.set_xlim(overall_min, overall_max)
    ax.set_ylim(overall_min, overall_max)
    
    # Square aspect ratio so that y=x appears at 45 degrees
    ax.set_aspect('equal', adjustable='box')

    r2 = r2_score(YY[:, i], Yhat_DD_GPCE[:, i])
    rmse = np.sqrt(mean_squared_error(YY[:, i], Yhat_DD_GPCE[:, i]))
    
    # Adjusted R²
    n = len(YY[:, i])  # number of samples
    k = basis_num  # number of predictors (basis functions)
    if n > k + 1:
        adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - k - 1))
    else:
        adj_r2 = r2  # same as R² if there are too few samples

    ax.set_title(f'Output {i+1} (RMSE: {rmse:.4f}, R²: {r2:.4f})')
    ax.set_xlabel(r'$\hat Y$')
    ax.set_ylabel(r'$Y$')
    ax.legend()

    r2_data.append(r2)
    rmse_data.append(rmse)
    adj_r2_data.append(adj_r2)

plt.tight_layout()
plt.show()

In [ ]:
true_means = YY.mean(axis=0)
true_stds = YY.std(axis=0)

pred_means = C3_DD[0,:]
pred_vars = [np.sum(C3_DD[1:, j] ** 2) for j in range(C3_DD.shape[1])]
pred_stds = np.sqrt(pred_vars)

# Collect the results in one DataFrame
comparison_df = pd.DataFrame({
    'Output': [f'Output {i+1}' for i in range(len(target_values))],
    'True Mean': true_means,
    'Predicted Mean': pred_means,
    'True Std': true_stds,
    'Predicted Std': pred_stds
})

comparison_df['Mean Error (%)'] = (np.abs(comparison_df['True Mean'] - comparison_df['Predicted Mean']) / np.abs(comparison_df['True Mean'])) * 100
comparison_df['Std Error (%)'] = (np.abs(comparison_df['True Std'] - comparison_df['Predicted Std']) / np.abs(comparison_df['True Std'])) * 100
# Print with six decimal places
print(comparison_df.round(6))

# Copy to the clipboard (tab-separated)
try:
    comparison_df.round(6).to_clipboard(sep='\t', index=False)
    print("\nData copied to the clipboard.")
    print("Paste with Ctrl+V.")
except:
    print("Copy to the clipboard failed.")


In [ ]:
# !!!! 25.09.24
# Pearson Correlation Coefficient Analysis between Test Output and GPCE Predicted Output

from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

print("📊 Pearson Correlation Analysis: Test Output vs GPCE Predicted Output")
print("="*80)

# 1. Pearson correlation coefficient of each output
correlation_results = []
p_values = []

for i in range(len(target_values)):
    # Pearson correlation coefficient and p-value
    corr_coeff, p_value = pearsonr(YY[:, i], Yhat_DD_GPCE[:, i])
    correlation_results.append(corr_coeff)
    p_values.append(p_value)
    
    print(f"Output {i+1}: r = {corr_coeff:.4f}, p-value = {p_value:.2e}")

# 2. Summary statistics of the correlation coefficients
print(f"\n📈 Correlation Summary:")
print(f"Average Correlation: {np.mean(correlation_results):.4f}")
print(f"Min Correlation: {np.min(correlation_results):.4f}")
print(f"Max Correlation: {np.max(correlation_results):.4f}")
print(f"Std Correlation: {np.std(correlation_results):.4f}")

# 3. Collect the correlation results in a DataFrame
correlation_df = pd.DataFrame({
    'Output': [f'Output {i+1}' for i in range(len(target_values))],
    'Pearson_r': correlation_results,
    'P_value': p_values,
    'R_squared': [r**2 for r in correlation_results],  # coefficient of determination (R²)
    'Correlation_Strength': ['Very Strong (r ≥ 0.9)' if abs(r) >= 0.9 
                            else 'Strong (0.7 ≤ r < 0.9)' if abs(r) >= 0.7
                            else 'Moderate (0.5 ≤ r < 0.7)' if abs(r) >= 0.5
                            else 'Weak (0.3 ≤ r < 0.5)' if abs(r) >= 0.3
                            else 'Very Weak (r < 0.3)' 
                            for r in correlation_results]
})

print(f"\n📋 Detailed Correlation Results:")
print(correlation_df.round(4))

# 4. Visualization - bar chart of the correlation coefficients
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart of the correlation coefficients
colors = ['red' if r < 0.5 else 'orange' if r < 0.7 else "#09FF00" if r < 0.9 else 'green' 
          for r in correlation_results]
ax1.bar(range(1, len(target_values)+1), correlation_results, color=colors, alpha=0.7)
ax1.set_title('Pearson Correlation Coefficients\n(Test Output vs GPCE Prediction)')
ax1.set_xlabel('Output Number')
ax1.set_ylabel('Pearson r')
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3)

# Threshold lines
ax1.axhline(y=0.9, color='green', linestyle='--', alpha=0.7, label='Very Strong (r ≥ 0.9)')
ax1.axhline(y=0.7, color='#09FF00', linestyle='--', alpha=0.7, label='Strong (r ≥ 0.7)')
ax1.axhline(y=0.5, color='orange', linestyle='--', alpha=0.7, label='Moderate (r ≥ 0.5)')
ax1.legend()

# Bar chart of the R² values
r_squared_values = [r**2 for r in correlation_results]
ax2.bar(range(1, len(target_values)+1), r_squared_values, color='skyblue', alpha=0.7)
ax2.set_title('Coefficient of Determination (R²)\n(Explained Variance)')
ax2.set_xlabel('Output Number')
ax2.set_ylabel('R²')
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 5. Correlation heat map (optional)
fig, ax = plt.subplots(figsize=(10, 8))

# Correlation matrix between the true and predicted values
correlation_matrix = np.corrcoef(YY.T, Yhat_DD_GPCE.T)
n_outputs = len(target_values)

# Extract the true-vs-predicted block (upper-right block)
actual_vs_pred_corr = correlation_matrix[:n_outputs, n_outputs:]

# Heat map
sns.heatmap(actual_vs_pred_corr, 
            annot=True, 
            cmap='RdYlGn', 
            center=0.8,
            vmin=0, vmax=1,
            xticklabels=[f'Pred_{i+1}' for i in range(n_outputs)],
            yticklabels=[f'Actual_{i+1}' for i in range(n_outputs)],
            ax=ax)
ax.set_title('Correlation Heatmap: Actual vs Predicted Outputs')
plt.tight_layout()
plt.show()


# 7. Summary of the model performance
strong_corr_count = sum(1 for r in correlation_results if abs(r) >= 0.7)
very_strong_corr_count = sum(1 for r in correlation_results if abs(r) >= 0.9)

print(f"\n🎯 Model Performance Summary:")
print(f"Very Strong Correlations (r ≥ 0.9): {very_strong_corr_count}/{len(target_values)} outputs")
print(f"Strong+ Correlations (r ≥ 0.7): {strong_corr_count}/{len(target_values)} outputs")
print(f"Overall Model Quality: {'Excellent' if very_strong_corr_count >= len(target_values)*0.8 else 'Good' if strong_corr_count >= len(target_values)*0.8 else 'Fair' if strong_corr_count >= len(target_values)*0.5 else 'Poor'}")

In [ ]:
# 📊 Model Performance Summary with Adjusted R²
print("📊 Model Performance Summary")
print("="*80)

# Collect the performance metrics in a DataFrame
performance_df = pd.DataFrame({
    'Output': [f'Output {i+1}' for i in range(len(target_values))],
    'R²': r2_data,
    'Adjusted_R²': adj_r2_data,
    'RMSE': rmse_data,
    'R²_Improvement': [adj_r2 - r2 for adj_r2, r2 in zip(adj_r2_data, r2_data)]
})

print("📈 Performance Metrics:")
print(performance_df.round(6))

# Summary statistics of the performance
print(f"\n📊 Summary Statistics:")
print(f"Average R²: {np.mean(r2_data):.6f}")
print(f"Average Adjusted R²: {np.mean(adj_r2_data):.6f}")
print(f"Best Adjusted R²: {np.max(adj_r2_data):.6f} (Output {np.argmax(adj_r2_data)+1})")
print(f"Worst Adjusted R²: {np.min(adj_r2_data):.6f} (Output {np.argmin(adj_r2_data)+1})")

# Performance grade based on the adjusted R²
excellent_count = sum(1 for adj_r2 in adj_r2_data if adj_r2 >= 0.9)
good_count = sum(1 for adj_r2 in adj_r2_data if 0.7 <= adj_r2 < 0.9)
fair_count = sum(1 for adj_r2 in adj_r2_data if 0.5 <= adj_r2 < 0.7)
poor_count = sum(1 for adj_r2 in adj_r2_data if adj_r2 < 0.5)

print(f"\n🎯 Performance Distribution (Adjusted R²):")
print(f"Excellent (≥ 0.9): {excellent_count}/{len(target_values)} outputs")
print(f"Good (0.7-0.9): {good_count}/{len(target_values)} outputs")
print(f"Fair (0.5-0.7): {fair_count}/{len(target_values)} outputs")
print(f"Poor (< 0.5): {poor_count}/{len(target_values)} outputs")


In [ ]:
# print("📊 Poster-style distribution comparison: RBDO vs DDO")
# print("=" * 80)

# from scipy import stats


# def finite_array(values):
#     values = np.asarray(values, dtype=float)
#     return values[np.isfinite(values)]


# fig, axes = plt.subplots(2, 2, figsize=(36, 18))
# axes = axes.flatten()

# plot_specs = [
#     {
#         "title": "",
#         "xlabel": "Mass",
#         "rbd_data": finite_array(Y_all_predicted[:, 0]),
#         "ddo_data": finite_array(Y_large_predicted_DDO[:, 0]),
#         "threshold": None,
#         "direction": None,
#         "xlim": None,
#     },
#     {
#         "title": "",
#         "xlabel": "Equivalent plastic strain",
#         "rbd_data": finite_array(Y_all_predicted[:, 1]),
#         "ddo_data": finite_array(Y_large_predicted_DDO[:, 1]),
#         "threshold": 0.05,
#         "direction": "above",
#         "xlim": (0.0, 0.15),
#     },
#     {
#         "title": "",
#         "xlabel": "X stiffness",
#         "rbd_data": finite_array(Y_all_predicted[:, 2]),
#         "ddo_data": finite_array(Y_large_predicted_DDO[:, 2]),
#         "threshold": 10,
#         "direction": "below",
#         "xlim": None,
#     },
#     {
#         "title": "",
#         "xlabel": "Y stiffness",
#         "rbd_data": finite_array(Y_all_predicted[:, 3]),
#         "ddo_data": finite_array(Y_large_predicted_DDO[:, 3]),
#         "threshold": 100,
#         "direction": "below",
#         "xlim": (90, None),
#     },
# ]

# for idx, spec in enumerate(plot_specs):
#     ax = axes[idx]
#     rbd_data = spec["rbd_data"]
#     ddo_data = spec["ddo_data"]

#     all_data = np.concatenate([rbd_data, ddo_data])
#     data_min = np.min(all_data)
#     data_max = np.max(all_data)
#     bins = np.linspace(data_min, data_max, 120)

#     ax.hist(ddo_data, bins=bins, density=True, alpha=0.60, color="#1976D2",
#             edgecolor="black", linewidth=0.6, label="DDO", zorder=2)
#     ax.hist(rbd_data, bins=bins, density=True, alpha=0.70, color="#C62828",
#             edgecolor="black", linewidth=0.6, label="RBDO", zorder=3)

#     if spec["threshold"] is not None:
#         threshold = spec["threshold"]
#         direction = spec["direction"]

#         if direction == "above":
#             ax.axvspan(threshold, data_max, alpha=0.25, color="#FFB6C1", zorder=0)
#             failure_prob_rbd = np.sum(rbd_data > threshold) / len(rbd_data) * 100
#             failure_prob_ddo = np.sum(ddo_data > threshold) / len(ddo_data) * 100
#         else:
#             failure_start = 90 if "Y stiffness" in spec["title"] else data_min
#             ax.axvspan(failure_start, threshold, alpha=0.25, color="#FFB6C1", zorder=0)
#             failure_prob_rbd = np.sum(rbd_data < threshold) / len(rbd_data) * 100
#             failure_prob_ddo = np.sum(ddo_data < threshold) / len(ddo_data) * 100

#         ax.axvline(threshold, color="#EF5350", linestyle="--", linewidth=2.0, zorder=4)
#         ax.set_title(
#             " ",
#             fontsize=60,
#             fontweight="bold",
#         )
#     else:
#         ax.set_title(spec["title"], fontsize=14, fontweight="bold")

#     ax.set_xlabel(spec["xlabel"], fontsize=25)
#     ax.set_ylabel(" ", fontsize=18)
#     ax.grid(True, alpha=0.3)
#     ax.tick_params(axis="both", labelsize=18)

#     if spec["xlim"] is not None:
#         x_left, x_right = spec["xlim"]
#         if x_left is None:
#             x_left = data_min
#         if x_right is None:
#             x_right = data_max
#         ax.set_xlim(x_left, x_right)

#     #ax.legend(fontsize=9, loc="upper right")

# plt.tight_layout()
# plt.savefig("poster_distribution_comparison_2x2.png", dpi=300, bbox_inches="tight")
# print("Saved to poster_distribution_comparison_2x2.png")
# plt.show()

In [ ]:
def calculate_cvar(data, beta=0.9):
    """
    Conditional Value at Risk (CVaR)
    
    Parameters:
    data: output data (1D array)
    beta: confidence level (0.9 = 90%)
    
    Returns:
    var: Value at Risk
    cvar: Conditional Value at Risk (exact formula)
    """
    # 1. VaR (quantile)
    var = np.percentile(data, beta * 100)
    
    # 2. CVaR (exact formula: mean of the values exceeding VaR divided by (1-beta))
    tail_values = data[data >= var]
    if len(tail_values) > 0:
        cvar = np.mean(tail_values)
        # Exact CVaR formula (theoretically divided by (1-beta),
        # but the mean of the tail is used in practice)
        # cvar = cvar / (1 - beta)  # theoretical formula
    else:
        cvar = var  # no tail
    
    return var, cvar

# CVaR comparison and results
print("📊 CVaR Comparison Analysis (MCS vs DD-GPCE)")
print("="*70)

beta = 0.9  # 90% confidence level

# List for the results
cvar_results = []

for i in range(len(target_values)):  # Output 1~9
    true_data = YY[:, i]
    pred_data = Yhat_DD_GPCE[:, i]
    
    # CVaR of the MCS data
    var_true, cvar_true = calculate_cvar(true_data, beta)
    
    # CVaR of the DD-GPCE predictions
    var_pred, cvar_pred = calculate_cvar(pred_data, beta)
    
    # Relative errors
    var_error = abs(var_pred - var_true) / abs(var_true) * 100 if var_true != 0 else 0
    cvar_error = abs(cvar_pred - cvar_true) / abs(cvar_true) * 100 if cvar_true != 0 else 0
    
    print(f"Output {i+1}:")
    print(f"  90% VaR  - MCS: {var_true:.3f}, DD-GPCE: {var_pred:.3f}, Error: {var_error:.2f}%")
    print(f"  90% CVaR - MCS: {cvar_true:.3f}, DD-GPCE: {cvar_pred:.3f}, Error: {cvar_error:.2f}%")
    print()
    
    # Store the results
    cvar_results.append({
        'Output': f'Output {i+1}',
        'VaR_MCS': var_true,
        'VaR_DD_GPCE': var_pred,
        'VaR_Error_%': var_error,
        'CVaR_MCS': cvar_true,
        'CVaR_DD_GPCE': cvar_pred,
        'CVaR_Error_%': cvar_error
    })

# Collect the results in a DataFrame
cvar_df = pd.DataFrame(cvar_results)
print("📊 CVaR Comparison Summary Table:")
print(cvar_df.round(3))


# Mean errors
avg_var_error = cvar_df['VaR_Error_%'].mean()
avg_cvar_error = cvar_df['CVaR_Error_%'].mean()

print(f"\n💡 Overall Performance:")
print(f"Average VaR Error: {avg_var_error:.2f}%")
print(f"Average CVaR Error: {avg_cvar_error:.2f}%")

# CVaR performance
if avg_cvar_error < 5:
    performance = "Excellent"
elif avg_cvar_error < 10:
    performance = "Good"
elif avg_cvar_error < 20:
    performance = "Fair"
else:
    performance = "Poor"

print(f"CVaR Prediction Performance: {performance}")

In [ ]:
# CVaR comparison plot
#fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig, axes = plt.subplots(3, 4, figsize=(32, 15))
axes = axes.flatten()

for i in range(len(target_values)):  # Output 1~9
    ax = axes[i]
    
    true_data = YY[:, i]
    pred_data = Yhat_DD_GPCE[:, i]
    
    # CVaR
    var_true, cvar_true = calculate_cvar(true_data, beta)
    var_pred, cvar_pred = calculate_cvar(pred_data, beta)
    
    # Use the same bin range
    all_data = np.concatenate([true_data, pred_data])
    data_min, data_max = np.min(all_data), np.max(all_data)
    bins = np.linspace(data_min, data_max, 80)
    
    # Step histograms (no overlap)
    ax.hist(true_data, bins=bins, alpha=0.7, color="#1e0eff", label='MCS', 
            density=True, histtype='step', linewidth=2.5)
    ax.hist(pred_data, bins=bins, alpha=0.7, color='#ff7f0e', label='DD-GPCE', 
            density=True, histtype='step', linewidth=2.5)
    
    # Draw the VaR lines
    ax.axvline(var_true, color='red', linestyle='--', linewidth=2, 
               label=f'VaR MCS: {var_true:.2f}')
    ax.axvline(var_pred, color='green', linestyle='--', linewidth=2, 
               label=f'VaR DD-GPCE: {var_pred:.2f}')
    
    # Draw the CVaR lines
    ax.axvline(cvar_true, color='red', linestyle='-', linewidth=3, 
               label=f'CVaR MCS: {cvar_true:.2f}')
    ax.axvline(cvar_pred, color='green', linestyle='-', linewidth=3, 
               label=f'CVaR DD-GPCE: {cvar_pred:.2f}')
    
    ax.set_title(f'Output {i+1} - VaR & CVaR Comparison (β={beta})')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    #ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Bar chart of the CVaR errors
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

outputs = [f'Output {i+1}' for i in range(len(target_values))]
var_errors = cvar_df['VaR_Error_%'].values
cvar_errors = cvar_df['CVaR_Error_%'].values

# VaR error
ax1.bar(outputs, var_errors, color='skyblue', alpha=0.7)
ax1.set_title('VaR Prediction Error (%)')
ax1.set_ylabel('Error (%)')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=5, color='red', linestyle='--', alpha=0.7, label='5% threshold')
ax1.legend()

# CVaR error
ax2.bar(outputs, cvar_errors, color='lightcoral', alpha=0.7)
ax2.set_title('CVaR Prediction Error (%)')
ax2.set_ylabel('Error (%)')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=5, color='red', linestyle='--', alpha=0.7, label='5% threshold')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Comparison of the distributions, means, and standard deviations
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
axes = axes.flatten()

# Number of bins
n_bins = 30

for i in range(len(target_values)):
    ax = axes[i]
    
    true_data = YY[:, i]
    pred_data = Yhat_DD_GPCE[:, i]
    
    # Use the same bin range (separately for each output)
    all_data = np.concatenate([true_data, pred_data])
    data_min, data_max = np.min(all_data), np.max(all_data)
    bins = np.linspace(data_min, data_max, n_bins + 1)
    
    # Means
    mean_true = np.mean(true_data)
    mean_pred = np.mean(pred_data)
    
    # Histograms: MCS as a blue line, DD-GPCE as orange fill
    ax.hist(true_data, bins=bins, alpha=1.0, color="#1e0eff", label='MCS', 
            density=True, histtype='step', linewidth=2.5)
    ax.hist(pred_data, bins=bins, alpha=0.8, color='#ff7f0e', label='DD-GPCE', 
            density=True, histtype='stepfilled', edgecolor='#ff7f0e')
    
    # Mean lines (thicker and darker)
    ax.axvline(mean_true, color='darkred', linestyle='-', linewidth=4, 
               label=f'MCS Mean: {mean_true:.2f}', zorder=10)  # draw in front
    ax.axvline(mean_pred, color='darkgreen', linestyle='--', linewidth=4, 
               label=f'DD-GPCE Mean: {mean_pred:.2f}', zorder=10)
    
    # Title and axis labels
    ax.set_title(f'Output {i+1}', fontsize=15, fontweight='bold')
    ax.set_xlabel('Value', fontsize=15)
    ax.set_ylabel('Density', fontsize=15)
    
    # Tick size
    ax.tick_params(axis='both', labelsize=15)
    
    ax.grid(True, alpha=0.3)
    
    # Smaller legend font and position
    #ax.legend(fontsize=7, loc='upper right', framealpha=0.9)

plt.tight_layout()

# Save as an image file
plt.savefig('mean_std_comparison.png', dpi=300, bbox_inches='tight')
print("Comparison of the distributions saved to 'mean_std_comparison.png'.")

plt.show()
